# Tính năng 2 – Trang **Tổng quan** (bức tranh thị trường)

Trang có 4 thẻ số liệu và 6 biểu đồ mô tả thị trường từ 1.735 tin tuyển dụng thật có ghi lương. Phần này chỉ **mô tả dữ liệu**, không dùng mô hình dự đoán (trừ thẻ "sai số"). Notebook tính lại từng con số, vẽ lại từng biểu đồ, kèm điều rút ra được và lý do chọn loại biểu đồ.

Mã nguồn: `pages/overview.py`, `src/career_market/ui/charts.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.data.dataset import load_dataset
from career_market.model.training import load_report

frame = load_dataset()
report = load_report()
facts = charts.summary(frame)
selected = report["models"][report["selected"]]
pd.Series({
    "Số tin có lương": f"{facts['postings']:,}", "Số công ty": facts["companies"],
    "Trung vị lương (triệu)": round(facts["median"], 1), "Trung bình (triệu)": round(facts["mean"], 1), "P90 (triệu)": round(facts["p90"], 1),
    "Top 3 công nghệ": " · ".join(charts.skill_table(frame).nlargest(3, "postings")["skill"]),
    "Số vai trò": facts["roles"], "Mô hình tốt nhất": selected["label"],
    "MAE test (triệu)": round(selected["test"]["MAE"], 2), "R² test": round(selected["test"]["R2"], 2),
}, name="KPI").to_frame()

### Vì sao dùng **trung vị** thay cho trung bình?
Một phòng có 9 người lương 20 triệu và một giám đốc lương 200 triệu: lương trung bình là 38 triệu, nhưng không ai trong 9 người kia nhận gần 38 triệu. Trung vị (20 triệu) mới mô tả đúng "người bình thường" trong phòng.

In [ ]:
team = pd.Series([20] * 9 + [200])
print(f"Ví dụ: trung bình {team.mean():.0f} · trung vị {team.median():.0f} triệu")
print(f"Dữ liệu thật: trung bình {frame['salary'].mean():.1f} · trung vị {frame['salary'].median():.1f} · độ lệch phải (skew) {frame['salary'].skew():.2f}")

## 1. Công nghệ được tuyển nhiều – và lương theo công nghệ
Dùng **biểu đồ cột nằm ngang** vì có nhiều kỹ năng và tên khá dài. Trang web hiện toàn bộ kỹ năng; ở đây chỉ vẽ 15 kỹ năng đầu cho gọn.

In [ ]:
charts.top_skill_demand(frame, limit=15);

In [ ]:
charts.top_skill_salary(frame, limit=15);

In [ ]:
table = charts.skill_table(frame)
pd.concat([
    table.nlargest(8, "postings")[["skill", "postings", "median"]].reset_index(drop=True).add_prefix("Phổ biến: "),
    table[table["postings"] >= 30].nlargest(8, "median")[["skill", "postings", "median"]].reset_index(drop=True).add_prefix("Lương cao (≥30 tin): "),
], axis=1)

**Điều rút ra:** kỹ năng được tuyển nhiều nhất (Java, React, SQL) không phải kỹ năng trả cao nhất (AWS, LLM/GenAI, Next.js). Đây mới chỉ là quan sát: tin đòi AWS có thể đơn giản là tin tuyển người nhiều kinh nghiệm. Mô hình ở tính năng 4 mới tách được hai yếu tố này.

## 2. Kinh nghiệm và thu nhập
Dùng **biểu đồ hộp theo số năm kinh nghiệm**, cộng một đường nối các trung vị, để thấy cả xu hướng lẫn độ chênh giữa các tin. Mức độ liên quan đo bằng **hệ số Spearman** (chỉ xét thứ tự cao/thấp), vì lương không tăng theo đường thẳng và có vài tin lương rất cao.

In [ ]:
from scipy.stats import spearmanr

known = frame.dropna(subset=["experience_years"])
print(f"Spearman ρ = {spearmanr(known['experience_years'], known['salary']).statistic:.2f} trên {len(known):,} tin có kinh nghiệm")
by_year = known.groupby(pd.cut(known["experience_years"], charts.EXPERIENCE_BINS, labels=charts.EXPERIENCE_LABELS), observed=False)["salary"]
display(pd.DataFrame({"Số tin": by_year.size(), "Trung vị (triệu)": by_year.median()}).T)
charts.experience_salary(frame);

Lương nhảy mạnh nhất giữa **4 và 5 năm** (từ Middle lên Senior): lên Senior là đổi vai trò, không chỉ thêm một năm làm việc.

## 3. Khu vực

In [ ]:
regions = frame.groupby("region")["salary"].agg(["size", "median"]).sort_values("size", ascending=False)
regions["Tỉ lệ tin"] = (regions["size"] / len(frame) * 100).round(1).astype(str) + "%"
display(regions.rename(columns={"size": "Số tin", "median": "Trung vị"}))
charts.region_salary(frame);

⚠️ Không nên kết luận "TP. Hồ Chí Minh lương thấp hơn": công ty ở TP. Hồ Chí Minh hay ghi "Thương lượng", nên số tin có lương ít và có thể không đại diện.

## 4. Vai trò, và vai trò theo từng khu vực
**Bản đồ nhiệt** (ô càng đậm, lương càng cao) cho bảng hai chiều vai trò × khu vực. Ô có dưới 3 tin được để trống có chủ đích, vì quá ít để tin.

In [ ]:
charts.role_salary(frame);

In [ ]:
charts.role_region_heatmap(frame);

## Kết luận
1. Kinh nghiệm là yếu tố ảnh hưởng mạnh nhất; bước nhảy lớn nhất là khi lên Senior.
2. Data Engineer, DevOps, quản lý kỹ thuật được trả cao hơn rõ so với lập trình phổ thông (nhưng vài nhóm có rất ít tin).
3. Kỹ năng được tuyển nhiều chưa chắc là kỹ năng trả cao.
4. So sánh giữa các khu vực luôn phải kèm cảnh báo về việc nhiều công ty không ghi lương.